# Problem 242: Odd Triplets

https://projecteuler.net/problem=242

Given the set $\{1,2,\dots,n\}$, we define $f(n, k)$ as the number of its $k$-element subsets with an odd sum of elements. For example, $f(5,3) = 4$, since the set $\{1,2,3,4,5\}$ has four $3$-element subsets having an odd sum of elements, i.e.: $\{1,2,4\}$, $\{1,3,5\}$, $\{2,3,4\}$ and $\{2,4,5\}$.

When all three values $n$, $k$ and $f(n, k)$ are odd, we say that they make an **odd-triplet** $[n,k,f(n, k)]$.

There are exactly five odd-triplets with $n \le 10$, namely:\
$[1,1,f(1,1) = 1]$, $[5,1,f(5,1) = 3]$, $[5,5,f(5,5) = 1]$, $[9,1,f(9,1) = 5]$ and $[9,9,f(9,9) = 1]$.

How many odd-triplets are there with $n \le 10^{12}$?

## Brute force

Directly enumerate all k-element subsets of {1, ..., n} using `itertools.combinations` and count those with an odd sum.

In [9]:
from itertools import combinations

def count_odd_sum_subsets(n: int, k: int) -> int:
    """Count k-element subsets of {1, ..., n} whose elements sum to an odd number."""
    return sum(1 for subset in combinations(range(1, n + 1), k) if sum(subset) % 2 == 1)

# Verify against the examples given in the problem
examples = [(5, 3, 4), (1, 1, 1), (5, 1, 3), (5, 5, 1), (9, 1, 5), (9, 9, 1)]
for n, k, expected in examples:
    result = count_odd_sum_subsets(n, k)
    status = "✓" if result == expected else "✗"
    print(f"count_odd_sum_subsets({n}, {k}) = {result}  (expected {expected})  {status}")

count_odd_sum_subsets(5, 3) = 4  (expected 4)  ✓
count_odd_sum_subsets(1, 1) = 1  (expected 1)  ✓
count_odd_sum_subsets(5, 1) = 3  (expected 3)  ✓
count_odd_sum_subsets(5, 5) = 1  (expected 1)  ✓
count_odd_sum_subsets(9, 1) = 5  (expected 5)  ✓
count_odd_sum_subsets(9, 9) = 1  (expected 1)  ✓


## Closed form via odd/even split

The key observation is that the sum of a subset is odd if and only if the subset contains an **odd number of odd elements** (since odd+odd=even, pairs of odds cancel, leaving only the parity of the count).

Split {1, ..., n} into `n_odd = ceil(n/2)` odd numbers and `n_even = floor(n/2)` even numbers. A k-element subset with exactly j odd elements contributes C(n_odd, j) * C(n_even, k-j) subsets. Summing over odd j:

```
count_odd_sum_subsets(n, k) = sum_{j odd, 1 <= j <= min(k, n_odd)} C(n_odd, j) * C(n_even, k-j)
```

This avoids enumerating all subsets and runs in O(k) time.

In [10]:
from math import comb

def count_odd_sum_subsets_closed_form(n: int, k: int) -> int:
    """Count k-element subsets of {1..n} with odd sum using the odd/even split identity."""
    n_odd = (n + 1) // 2
    n_even = n // 2
    return sum(
        comb(n_odd, j) * comb(n_even, k - j)
        for j in range(1, min(k, n_odd) + 1, 2)
        if k - j <= n_even
    )

# Verify it matches the brute force for all valid (n, k) up to n=20
mismatches = [
    (n, k) for n in range(1, 21) for k in range(1, n + 1)
    if count_odd_sum_subsets(n, k) != count_odd_sum_subsets_closed_form(n, k)
]
if mismatches:
    print(f"Mismatches: {mismatches}")
else:
    print("Closed form matches brute force for all n ≤ 20  ✓")

Closed form matches brute force for all n ≤ 20  ✓


## When is count_odd_sum_subsets(n, k) odd?

We want to know when the value of the sum is itself odd. Let `n_odd = ceil(n/2)` and `n_even = floor(n/2)`.

**Claim:** `count_odd_sum_subsets(n, k)` is odd if and only if:
1. `n_odd` is odd, **and**
2. `C(n-1, k-1)` is odd

**Proof sketch:**

- **If `n_odd` is even:** By Lucas' theorem, `C(n_odd, j)` is even for every odd `j` (last bit of `n_odd` is 0, last bit of odd `j` is 1). Every term in the sum vanishes mod 2, so the total is even.

- **If `n_odd` is odd:** Apply Pascal's identity `C(n_odd, j) = C(n_odd-1, j) + C(n_odd-1, j-1)`. Since `n_odd-1` is even and `j` is odd, `C(n_odd-1, j)` is even by Lucas, so `C(n_odd, j) ≡ C(n_odd-1, j-1) mod 2`. Re-index with `r = j-1` (r is even). The odd-r terms also vanish (same Lucas argument: `n_odd-1` even, `r` odd). Extending the sum to all r and applying Vandermonde's identity gives `C(n_odd + n_even - 1, k-1) = C(n-1, k-1) mod 2`.

**Lucas criterion for C(n-1, k-1):** By Lucas' theorem, `C(n-1, k-1)` is odd iff every 1-bit of `k-1` is also a 1-bit of `n-1`. Equivalently (by Kummer's theorem), this holds iff adding `k-1` and `n-k` in binary produces no carries, i.e. `(k-1) & (n-k) == 0`.

In [11]:
def is_odd_triplet(n: int, k: int) -> bool:
    """Return True if [n, k, count_odd_sum_subsets(n, k)] is an odd-triplet.
    
    Requires n odd, k odd, n_odd = ceil(n/2) odd, and (k-1) & (n-k) == 0.
    """
    if n % 2 == 0 or k % 2 == 0:
        return False
    n_odd = (n + 1) // 2
    if n_odd % 2 == 0:
        return False
    return (k - 1) & (n - k) == 0

# Verify against closed form for all odd (n, k) up to n=30
mismatches = [
    (n, k) for n in range(1, 31) for k in range(1, n + 1)
    if (n % 2 == 1 and k % 2 == 1)
    and (count_odd_sum_subsets_closed_form(n, k) % 2 == 1) != is_odd_triplet(n, k)
]
if mismatches:
    print(f"Mismatches: {mismatches}")
else:
    print("Parity criterion matches closed form for all n ≤ 30  ✓")

first_ten = [(n, k) for n in range(1, 50) for k in range(1, n + 1) if is_odd_triplet(n, k)][:10]
print(f"First 10 odd-triplets (n, k): {first_ten}")

Parity criterion matches closed form for all n ≤ 30  ✓
First 10 odd-triplets (n, k): [(1, 1), (5, 1), (5, 5), (9, 1), (9, 9), (13, 1), (13, 5), (13, 9), (13, 13), (17, 1)]


## Counting all odd-triplets up to n ≤ N

The three conditions on an odd-triplet reduce to:
1. `n ≡ 1 (mod 4)` — the last two bits of n must be `01`. This simultaneously ensures n is odd and `n_odd = ceil(n/2)` is odd. (If n = 4q+1 then n_odd = 2q+1, which is odd.)
2. `(k-1) & (n-k) == 0` — k-1 is a binary submask of n-1.

**All submasks are free:** Since n-1 ends in `00` (n ≡ 1 mod 4), all submasks of n-1 are even, making `k = submask + 1` automatically odd — no extra filtering needed. Each qualifying n contributes exactly `2^popcount(n-1)` valid k values.

**Reducing to a sum:** Writing n = 4m+1, so n-1 = 4m and popcount(4m) = popcount(m), the total count is:

```
sum_{m=0}^{floor((N-1)/4)}  2^popcount(m)
```

**Digit DP:** To evaluate `sum_{m=0}^{M} 2^popcount(m)` in O(log M), process M's bits from high to low. When bit i of M is 1, we can set it to 0 and freely choose the lower i bits — each free bit either contributes 0 ones (weight 1) or 1 one (weight 2), so i free bits multiply by 3^i. Tracking `ones_above` (number of 1-bits seen so far), the contribution of each such free prefix is `2^ones_above * 3^i`.

In [12]:
def count_odd_triplets(N: int) -> int:
    """Count odd-triplets [n, k, f(n,k)] with n <= N using digit DP on sum_{m=0}^{M} 2^popcount(m)."""
    M = (N - 1) // 4
    total = 0
    ones_above = 0
    for i in range(M.bit_length() - 1, -1, -1):
        if (M >> i) & 1:
            total += (2 ** ones_above) * (3 ** i)
            ones_above += 1
    total += 2 ** ones_above  # count M itself
    return total

# Verify against brute force
def count_odd_triplets_brute(N: int) -> int:
    return sum(1 for n in range(1, N + 1) for k in range(1, n + 1) if is_odd_triplet(n, k))

for N in [10, 100, 1000]:
    fast, brute = count_odd_triplets(N), count_odd_triplets_brute(N)
    status = "✓" if fast == brute else "✗"
    print(f"N={N:<6}  count={fast:<8}  brute={brute:<8}  {status}")

print()
print(f"Answer for n ≤ 10^12:  {count_odd_triplets(10**12)}")

N=10      count=5         brute=5         ✓
N=100     count=139       brute=139       ✓
N=1000    count=5793      brute=5793      ✓

Answer for n ≤ 10^12:  997104142249036713
